In [0]:
%sql
use catalog training_dev;

#### Setup
- create volume
- create folder within volume
- upload file to volume

In [0]:
%sql
create volume if not exists training_dev.default.raw;

In [0]:
%fs
mkdirs /Volumes/training_dev/default/raw/order_hist

In [0]:
%fs
ls /Volumes/training_dev/default/raw

In [0]:
dbutils.fs.ls('/Volumes/training_dev/default/raw/order_hist/')

In [0]:
%sql
select * from read_files('dbfs:/Volumes/training_dev/default/raw/order_hist',format=>'csv')
limit 3;

##### Ingestion
- **Incremental Load:** Use Copy Into
- Running copy into many times does not insert the same file data again.
- table history or version - no change

In [0]:
%sql

drop table if exists training_dev.default.order_m ;
create table training_dev.default.order_m (order_id int, order_date timestamp, order_status string, customer_id string, _file_name string, _load_dt timestamp)
;

In [0]:
%sql
select * from order_m;

##### Upload another file - same schema and datatypes
- ONLY incremental processing

In [0]:
%sql
COPY INTO training_dev.default.order_m
from ( select 
             cast(order_id as integer) as order_id,
             cast(order_date as timestamp) as order_date,
             order_status,
             customer_id,
             _metadata.file_name as _file_name,
             current_timestamp() as _load_dt
        from
            '/Volumes/training_dev/default/raw/order_hist/'
      )
FILEFORMAT = CSV
FORMAT_OPTIONS ("header" = "true")
COPY_OPTIONS ("mergeSchema" = "true")
;

select * from training_dev.default.order_m;

##### Upload another file, with an extra column and run COPY INTO again
- Extra column is NOT reflected as the "SELECT" has selective columns

In [0]:
%sql
COPY INTO training_dev.default.order_m
from ( select 
             cast(order_id as integer) as order_id,
             cast(order_date as timestamp) as order_date,
             order_status,
             customer_id,
             _metadata.file_name as _file_name,
             current_timestamp() as _load_dt
        from
            '/Volumes/training_dev/default/raw/order_hist/'
      )
FILEFORMAT = CSV
FORMAT_OPTIONS ("header" = "true")
COPY_OPTIONS ("mergeSchema" = "true")
;

select * from training_dev.default.order_m;

##### Upload another file, with an incorrect datatype and run COPY INTO again
- incremental processing ONLY

In [0]:
%sql
COPY INTO training_dev.default.order_m
from ( select 
             cast(order_id as integer) as order_id,
             cast(order_date as timestamp) as order_date,
             order_status,
             customer_id,
             _metadata.file_name as _file_name,
             current_timestamp() as _load_dt
        from
            '/Volumes/training_dev/default/raw/order_hist/'
      )
FILEFORMAT = CSV
FORMAT_OPTIONS ("header" = "true")
--COPY_OPTIONS ("mergeSchema" = "true")
;

select * from training_dev.default.order_m;